# ISOM5240 Fine-tuning Notebook — Pipeline 2: Freshness Detection

**Workflow:**
1. Phase 1 → Evaluate 3 pre-trained models (NO fine-tuning)
2. Phase 2 → Select the best one
3. Phase 3 → Fine-tune ONLY the selected model
4. Phase 4 → Compare before vs after fine-tuning
5. Phase 5 → Push fine-tuned model to HuggingFace Hub

**Dataset:** Kaggle "Fruits Fresh and Rotten for Classification"
- URL: https://www.kaggle.com/datasets/sriramr/fruits-fresh-and-rotten-for-classification
- 13,599 images, 6 classes → re-label to 2 (fresh / rotten)

## Step 1: Install dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate pillow -q
!pip install huggingface_hub -q

## Step 2: Login to HuggingFace Hub

Uses Colab Secrets (🔑 icon in left sidebar). Add `HF_TOKEN` there before running.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("Logged in to HuggingFace Hub via Colab Secrets")

## Step 3: Download dataset from Kaggle

Uses Colab Secrets (🔑 icon in left sidebar). Add `KAGGLE_USERNAME` and `KAGGLE_KEY` there before running.

(Get them from kaggle.com → Account → Create New API Token → open the downloaded kaggle.json)

In [ ]:
import os
from google.colab import userdata

# Setup Kaggle credentials from Colab Secrets
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

# Download and unzip the dataset
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification
!unzip -q fruits-fresh-and-rotten-for-classification.zip -d dataset/

print("Dataset downloaded!")
print("Folder structure:")
!find dataset/ -type d | head -20

## Step 4: Load dataset and re-label to binary

Original 6 classes:
- freshapples, freshbanana, freshoranges → **fresh (0)**
- rottenapples, rottenbanana, rottenoranges → **rotten (1)**

In [ ]:
from datasets import load_dataset, DatasetDict
import numpy as np
import time

# Load from local folder
dataset = load_dataset("imagefolder", data_dir="dataset/dataset")

print("Loaded dataset:")
print(dataset)
print(f"\nOriginal labels: {dataset['train'].features['label'].names}")
print(f"Train size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

In [ ]:
# Re-label: 6 classes → 2 classes (fresh=0, rotten=1)
original_names = dataset["train"].features["label"].names
print(f"Original class names: {original_names}")

label_mapping = {}
for idx, name in enumerate(original_names):
    if "fresh" in name.lower():
        label_mapping[idx] = 0  # fresh
    else:
        label_mapping[idx] = 1  # rotten

print(f"Label mapping: {label_mapping}")

def relabel_to_binary(example):
    example["label"] = label_mapping[example["label"]]
    return example

dataset = dataset.map(relabel_to_binary)

LABEL_NAMES = ["fresh", "rotten"]

# Count distribution
train_labels = np.array(dataset["train"]["label"])
test_labels = np.array(dataset["test"]["label"])
print(f"\nAfter re-labeling:")
print(f"Train: {len(train_labels)} total | fresh: {(train_labels==0).sum()} | rotten: {(train_labels==1).sum()}")
print(f"Test:  {len(test_labels)} total | fresh: {(test_labels==0).sum()} | rotten: {(test_labels==1).sum()}")

## Step 5: Split validation set from training data

Split 10% of train as validation for monitoring during training.

In [ ]:
split = dataset["train"].train_test_split(test_size=0.1, seed=42, stratify_by_column="label")
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": dataset["test"],
})

print(f"Final splits:")
print(f"  Train:      {len(dataset['train'])}")
print(f"  Validation: {len(dataset['validation'])}")
print(f"  Test:       {len(dataset['test'])}")

---
# PHASE 1: Evaluate 3 pre-trained models (NO fine-tuning)

Compare ViT, ResNet, Swin on inference speed and confidence to select the best candidate.

## Step 6: Define candidate models

In [ ]:
CANDIDATE_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

print("Candidate models for Phase 1 comparison:")
for name, path in CANDIDATE_MODELS.items():
    print(f"  {name}: {path}")

## Step 7: Evaluate each pre-trained model (zero-shot)

These models output ImageNet labels, not fresh/rotten. We compare parameter count, inference speed, and confidence distribution to determine which model's features are best suited for our task.

In [ ]:
from transformers import pipeline as hf_pipeline
import pandas as pd

def evaluate_pretrained(model_key, model_path, test_data, num_samples=200):
    """Evaluate a pre-trained model WITHOUT fine-tuning."""
    print(f"\nEvaluating: {model_key} ({model_path})")

    pipe = hf_pipeline("image-classification", model=model_path, device=0)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    n = min(num_samples, len(test_data))
    inference_times = []
    confidences = []

    for i in range(n):
        img = test_data[i]["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        t0 = time.time()
        result = pipe(img, top_k=1)
        inference_times.append(time.time() - t0)
        confidences.append(result[0]["score"])

    avg_time_ms = np.mean(inference_times) * 1000
    avg_confidence = np.mean(confidences)

    result = {
        "Model": model_key,
        "Parameters": total_params,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Avg Inference (ms)": round(avg_time_ms, 1),
        "Avg Confidence": round(avg_confidence, 4),
        "Samples Tested": n,
    }

    print(f"  Params: {total_params/1e6:.1f}M | Speed: {avg_time_ms:.1f}ms | Conf: {avg_confidence:.4f}")
    return result

In [ ]:
# Run Phase 1 evaluation
pretrained_results = []

for key, path in CANDIDATE_MODELS.items():
    r = evaluate_pretrained(key, path, dataset["test"])
    pretrained_results.append(r)

df_pretrained = pd.DataFrame(pretrained_results)
print("\n" + "="*60)
print("PHASE 1 RESULTS: Pre-trained Model Comparison")
print("="*60)
print(df_pretrained[["Model", "Parameters (M)", "Avg Inference (ms)", "Avg Confidence"]].to_string(index=False))

## Step 8: Select the best model

Based on Phase 1 results, pick the model with the best balance of feature quality and speed.

**⚠️ Update `SELECTED_MODEL_KEY` below after reviewing Phase 1 results.**

In [ ]:
# TODO: Update this based on your actual Phase 1 results
SELECTED_MODEL_KEY = "ViT-base"  # Change if ResNet or Swin performed better
SELECTED_MODEL_PATH = CANDIDATE_MODELS[SELECTED_MODEL_KEY]

print(f"\n>>> Selected model for fine-tuning: {SELECTED_MODEL_KEY}")
print(f">>> Path: {SELECTED_MODEL_PATH}")

---
# PHASE 2: Fine-tune the selected model

## Step 9: Preprocessing for the selected model

In [ ]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(SELECTED_MODEL_PATH)

def preprocess(example):
    image = example["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    example["pixel_values"] = inputs["pixel_values"].squeeze(0)
    return example

train_processed = dataset["train"].map(preprocess, remove_columns=["image"])
val_processed = dataset["validation"].map(preprocess, remove_columns=["image"])
test_processed = dataset["test"].map(preprocess, remove_columns=["image"])

print(f"Preprocessing done with {SELECTED_MODEL_KEY} processor")
print(f"Image size: {processor.size}")

## Step 10: Load model with 2-class classification head

In [ ]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    SELECTED_MODEL_PATH,
    num_labels=len(LABEL_NAMES),
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {SELECTED_MODEL_KEY}")
print(f"Total params: {total_params:,} | Trainable: {trainable:,}")

## Step 11: Training configuration

In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir=f"./freshness-{SELECTED_MODEL_KEY.lower()}",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    remove_unused_columns=False,
    push_to_hub=False,
)

print("Training config: 5 epochs, lr=2e-5, batch=16")

## Step 12: Train!

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    compute_metrics=compute_metrics,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start

print(f"\nTraining complete in {train_time/60:.1f} minutes")

## Step 13: Evaluate fine-tuned model on test set

In [ ]:
# Test accuracy
ft_results = trainer.evaluate(test_processed)
print(f"Fine-tuned Test Accuracy: {ft_results['eval_accuracy']:.4f}")
print(f"Fine-tuned Test Loss:     {ft_results['eval_loss']:.4f}")

# Inference speed after fine-tuning
pipe_ft = hf_pipeline("image-classification", model=model, image_processor=processor, device=0)
ft_times = []
for i in range(min(100, len(dataset["test"]))):
    img = dataset["test"][i]["image"]
    if img.mode != "RGB":
        img = img.convert("RGB")
    t0 = time.time()
    pipe_ft(img)
    ft_times.append(time.time() - t0)
ft_avg_ms = np.mean(ft_times) * 1000

print(f"Fine-tuned Avg Inference: {ft_avg_ms:.1f}ms")

In [ ]:
# Summary: Before vs After fine-tuning
selected_pretrained = next(r for r in pretrained_results if r["Model"] == SELECTED_MODEL_KEY)

comparison = pd.DataFrame([
    {
        "Stage": "Pre-trained (before)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": "N/A (ImageNet labels)",
        "Loss": "N/A",
        "Avg Inference (ms)": selected_pretrained["Avg Inference (ms)"],
    },
    {
        "Stage": "Fine-tuned (after)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": f"{ft_results['eval_accuracy']:.4f}",
        "Loss": f"{ft_results['eval_loss']:.4f}",
        "Avg Inference (ms)": round(ft_avg_ms, 1),
    },
])

print("\n" + "="*60)
print("PHASE 2 RESULTS: Before vs After Fine-tuning")
print("="*60)
print(comparison.to_string(index=False))

## Step 14: Save experiment results to Excel

In [ ]:
with pd.ExcelWriter("Experimental_results.xlsx") as writer:
    df_pretrained[["Model", "Parameters (M)", "Avg Inference (ms)", "Avg Confidence"]].to_excel(
        writer, sheet_name="Model Selection", index=False
    )
    comparison.to_excel(
        writer, sheet_name="Fine-tune Result", index=False
    )

print("Saved to Experimental_results.xlsx")

## Step 15: Push fine-tuned model to HuggingFace Hub

In [ ]:
HUB_MODEL_ID = "your-username/freshness-detection"  # TODO: Change to your username

trainer.push_to_hub(
    repo_id=HUB_MODEL_ID,
    commit_message=f"Fine-tuned {SELECTED_MODEL_KEY} for fruit freshness detection (acc={ft_results['eval_accuracy']:.4f})"
)
processor.push_to_hub(HUB_MODEL_ID)

print(f"\nModel pushed to: https://huggingface.co/{HUB_MODEL_ID}")
print("Use this URL in your report under 'Model URL'")

## Step 16: Final inference test

In [ ]:
pipe_final = hf_pipeline("image-classification", model=HUB_MODEL_ID)

for i in [0, 1, 2, 3, 4]:
    img = dataset["test"][i]["image"]
    true_label = LABEL_NAMES[dataset["test"][i]["label"]]
    pred = pipe_final(img)
    pred_label = pred[0]["label"]
    pred_score = pred[0]["score"]
    status = "correct" if pred_label == true_label else "WRONG"
    print(f"  [{status}] True: {true_label} | Predicted: {pred_label} ({pred_score:.3f})")